# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanmayshukla28/Flyrank-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pwd

/content


In [2]:
!ls

sample_data


In [3]:
!find . -maxdepth 2 -type d

.
./.config
./.config/logs
./.config/configurations
./sample_data


In [4]:
!git clone https://github.com/tanmayshukla28/Flyrank-internship-starter.git

Cloning into 'Flyrank-internship-starter'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 143 (delta 53), reused 96 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.86 MiB | 5.49 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [5]:
%cd Flyrank-internship-starter

/content/Flyrank-internship-starter


In [6]:
!ls

AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [7]:
!find . -maxdepth 2 -type d

.
./notebooks
./skills
./skills/writing-data-contracts
./skills/querying-big-datasets
./skills/training-honest-models
./skills/writing-research-papers
./skills/flyrank
./skills/deploying-static-pages
./skills/framing-ml-problems
./skills/writing-honest-claims
./skills/directing-your-ai-assistant
./skills/hunting-leakage-and-validating
./skills/building-baselines
./skills/auditing-signals
./.github
./.github/workflows
./data
./data/raw
./.git
./.git/info
./.git/logs
./.git/objects
./.git/refs
./.git/hooks
./.git/branches
./submission
./docs
./outputs
./outputs/charts
./work
./work/notebooks
./scripts


In [8]:
!cat skills/README.md

# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame your question (ML-02, ML-03) | `framing-ml-problems/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Write +

In [9]:
!ls skills/building-baselines

SKILL.md


In [10]:
!ls work/notebooks

capstone.ipynb			 w04_baseline_score.ipynb
w01_research_question.ipynb	 w04_signal_audit.ipynb
w02_ml_task_framing.ipynb	 w05_model.ipynb
w03_data_contract.ipynb		 w06_validation_audit.ipynb
w03_feature_leakage_check.ipynb  w07_action_playbook.ipynb


In [11]:
!head -100 skills/building-baselines/SKILL.md

---
name: building-baselines
description: Builds the transparent rule-based baseline every model must beat — a hand-written score with reason codes, ranked output, and precision@K evaluation. Use before training any model, or when someone reports model results with nothing to compare against.
---

# Building baselines

A model without a baseline is a number without a meaning. The baseline is a rule a human can
read — and its job is to be honestly beatable.

## Build it in this order

**1. Say the rule in plain words first.** "A page is worth reviewing if it used to get traffic,
it's getting old, and its position is slipping." If you can't say it, you can't code it.

**2. Code it as a transparent score.** Multiply/add simple conditions; no fitted weights:

```python
stale   = (df["days_since_update"] >= 180).astype(int)
visible = (df["impressions"] >= 500).astype(int)
df["score"] = stale * visible * df["impressions"]     # readable on purpose
```

**3. Attach reason codes.** Every score

In [12]:
import pandas as pd
import os

for root, dirs, files in os.walk("data"):
    for f in files:
        if f.endswith(".csv") or f.endswith(".parquet"):
            print(os.path.join(root, f))

data/raw/content_refresh_anonymized.csv


In [13]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.columns.tolist())
df.head()

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [14]:
for c in df.columns:
    print(c)

content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct


## 1. My rule and its reason codes

My baseline prioritizes content that has high search potential but is underperforming. Pages with high search volume, low CTR, old content, or declining trends receive higher scores because they are stronger candidates for optimization. Healthy pages receive lower scores.

In [15]:
score = (
    (df["search_volume"].fillna(0) > 50).astype(int) * 30 +
    (df["ctr"].fillna(0) < 0.03).astype(int) * 30 +
    (df["days_since_last_update"].fillna(0) > 180).astype(int) * 20 +
    (df["trend_pct"].fillna(0) < 0).astype(int) * 20
)

df["baseline_score"] = score

## 2. Build the ranked queue (writes the CSV)

The queue ranks pages using a simple rule-based score. Pages with higher scores are considered higher priority for review and optimization.

In [16]:
df["reason_code"] = "OK"

df.loc[df["ctr"] < 0.03, "reason_code"] = "LOW_CTR"
df.loc[df["days_since_last_update"] > 180, "reason_code"] = "STALE"
df.loc[df["trend_pct"] < 0, "reason_code"] = "DECLINING"

df = df.sort_values("baseline_score", ascending=False)

df.to_csv("outputs/baseline_action_score.csv", index=False)

df.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code
21984,content_02b0d6e30129,client_19581e27de,110.0,0.40,MEDIUM,0.59,keyword article,transactional,NaN,NaN,...,6.9,0.00,0.00,0.0,low,page_1,down,-95.6,100,DECLINING
7509,content_7a888d3d99c8,client_19581e27de,90.0,0.46,MEDIUM,0.72,keyword article,transactional,NaN,NaN,...,67.6,0.00,0.00,0.0,low,deep,down,-100.0,100,DECLINING
23619,content_24abafed9707,client_8722616204,480.0,0.00,LOW,0.00,keyword article,transactional,3246.0,21393.0,...,1.3,0.00,50.00,0.0,low,top_3,down,-100.0,100,DECLINING
16417,content_f4b3081037b3,client_8722616204,90.0,0.00,LOW,0.00,keyword article,transactional,3073.0,20510.0,...,5.0,0.00,0.00,0.0,low,page_1,down,-100.0,100,DECLINING
1659,content_bbca724138f2,client_6208ef0f77,1600.0,0.43,MEDIUM,3.76,keyword article,transactional,5614.0,37325.0,...,12.1,0.00,0.00,0.0,low,striking,down,-100.0,100,DECLINING
21583,content_1c8dd579b002,client_19581e27de,260.0,0.04,LOW,0.00,keyword article,informational,NaN,NaN,...,72.8,0.00,0.00,0.0,low,deep,down,-92.6,80,DECLINING
4331,content_6a8c4838e1ae,client_8527a891e2,90.0,0.30,LOW,0.07,keyword article,commercial,1622.0,10102.0,...,5.5,0.00,25.00,0.0,low,page_1,down,-100.0,80,DECLINING
4330,content_efb970557302,client_8722616204,70.0,0.20,LOW,1.82,keyword article,informational,2805.0,19206.0,...,18.2,0.00,66.67,0.0,low,striking,down,-100.0,80,DECLINING
21658,content_b1a4664c661b,client_19581e27de,90.0,0.43,MEDIUM,2.35,keyword article,informational,NaN,NaN,...,16.9,0.00,0.00,0.0,moderate,striking,stable,-14.7,80,DECLINING
4319,content_2c62e3f7b08a,client_d029fa3a95,90.0,0.29,LOW,0.00,keyword article,informational,4369.0,28001.0,...,10.5,0.00,50.00,0.0,low,striking,down,-100.0,80,DECLINING


## 3. Top-20 review

The highest ranked pages generally have low CTR, older content, or declining trends. These pages are good candidates for refreshing content or improving metadata. Some recommendations may be unnecessary if recent improvements have not yet appeared in the collected data.

In [17]:
df[[
    "content_id",
    "baseline_score",
    "reason_code",
    "ctr",
    "days_since_last_update",
    "trend_pct"
]].head(20)

,content_id,baseline_score,reason_code,ctr,days_since_last_update,trend_pct
21984,content_02b0d6e30129,100,DECLINING,0.0,313,-95.6
7509,content_7a888d3d99c8,100,DECLINING,0.0,313,-100.0
23619,content_24abafed9707,100,DECLINING,0.0,231,-100.0
16417,content_f4b3081037b3,100,DECLINING,0.0,231,-100.0
1659,content_bbca724138f2,100,DECLINING,0.0,236,-100.0
21583,content_1c8dd579b002,80,DECLINING,0.0,22,-92.6
4331,content_6a8c4838e1ae,80,DECLINING,0.0,8,-100.0
4330,content_efb970557302,80,DECLINING,0.0,8,-100.0
21658,content_b1a4664c661b,80,DECLINING,0.0,22,-14.7
4319,content_2c62e3f7b08a,80,DECLINING,0.0,8,-100.0


## 4. Weak picks + leakage check

Some low-priority pages may appear because of temporary traffic fluctuations rather than real problems. No future information or target labels were used while computing the baseline score, so the rule avoids data leakage.

In [18]:
df.tail(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code
1888,content_e0e1c06985b2,client_6208ef0f77,0.0,0.00,LOW,0.0,keyword article,informational,3552.0,22357.0,...,28.2,0.00,0.00,0.00,low,page_3_5,up,67.3,0,OK
26270,content_c69f0baaa1af,client_4e07408562,0.0,0.00,LOW,0.0,keyword article,informational,2861.0,18559.0,...,10.5,0.00,7.69,0.00,moderate,striking,up,26.9,0,OK
26260,content_1c6462dc651c,client_2c624232cd,10.0,0.00,LOW,0.0,keyword article,informational,NaN,NaN,...,48.8,15.00,70.00,0.00,low,page_3_5,up,260.0,0,OK
29970,content_4998a1c76243,client_f369cb89fc,0.0,0.00,LOW,0.0,keyword article,informational,3146.0,20309.0,...,7.1,0.00,0.00,0.00,moderate,page_1,up,90.7,0,OK
29968,content_179533212cd0,client_f74efabef1,30.0,0.02,LOW,0.0,keyword article,commercial,3884.0,25797.0,...,11.8,17.65,23.53,0.00,moderate,striking,new,NaN,0,OK
31,content_24ee79621dbf,client_19581e27de,50.0,0.09,LOW,0.0,keyword article,informational,NaN,NaN,...,42.0,0.00,0.00,0.00,moderate,page_3_5,up,42.5,0,OK
29999,content_887020f20b5e,client_6208ef0f77,0.0,0.00,LOW,0.0,keyword article,commercial,4124.0,26315.0,...,8.4,6.58,3.31,2.63,good,page_1,stable,12.3,0,OK
29985,content_fcad8c75dc44,client_bbb965ab0c,10.0,0.00,LOW,0.0,keyword article,informational,2666.0,19294.0,...,6.8,4.21,4.17,0.00,good,page_1,stable,6.7,0,OK
15,content_689414059706,client_9f14025af0,0.0,0.00,LOW,0.0,keyword article,informational,2661.0,17889.0,...,7.8,33.33,26.67,0.00,low,page_1,up,675.0,0,OK
12,content_42fb2cad9ecf,client_6208ef0f77,0.0,0.00,LOW,0.0,keyword article,informational,3969.0,24527.0,...,5.6,3.43,2.61,0.00,good,page_1,up,277.8,0,OK


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.